# Lesson 02 Lab — The Sparsity Granularity Spectrum: Weights, Channels, Blocks, and N:M

**Puzzle:** Can two tensors with exactly 50% zeros demand different kernels and deployment formats?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

The word sparsity hides a layout contract. Unstructured zeros, contiguous blocks, removed channels, and 2:4 groups can share the same global nonzero rate while exposing very different metadata, vectorization, and library opportunities. Choosing a granularity is therefore a joint algorithm-runtime decision, not a cosmetic choice made after training.


## 0. Predict before running

1. Predict which 50% mask will pass an exact 2:4 compliance check.
2. Predict whether ordinary dense matmul notices unstructured or block zeros.
3. Choose a granularity when custom kernels are forbidden.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The lab uses one weight matrix and derives four representations: unstructured magnitude zeros, block zeros, exact 2:4 groups, and a physically narrowed matrix. It tracks global sparsity, 2:4 compliance, shape, and dense-path latency.

- Equal nonzero counts do not imply equal layouts.
- N:M compliance is a local invariant, not a global percentage.
- Channel removal can use a smaller dense operator without sparse metadata.


## 2. Derive the mechanism

A global rate `1 - nnz/numel` discards where the nonzeros live. For 2:4 sparsity, every consecutive group of four along the contracted dimension must contain exactly two retained values; 50% zeros placed elsewhere are non-compliant. Block sparsity adds a block shape and index structure. Channel pruning removes a complete axis and can reuse dense kernels at a smaller dimension. Runtime value comes from matching one of these contracts to an implementation.

### Mechanism at a glance

```mermaid
flowchart TD
  Z["Same 50% zero budget"] --> U["Unstructured zeros<br/>same shape"]
  Z --> B["Block sparsity<br/>same shape + block metadata"]
  Z --> N["2:4 sparsity<br/>local pattern contract"]
  Z --> C["Channel pruning<br/>smaller physical shape"]
  U --> K["Runtime support decides value"]
  B --> K
  N --> K
  C --> K
```

### Walk it step by step

1. **Hold the zero budget fixed.** Compare layouts at the same global sparsity so granularity is the independent variable.
2. **Check the local contract.** Block and N:M layouts require local grouping rules that a global percentage cannot express.
3. **Check physical shape.** Channel removal changes dimensions and can reuse ordinary dense kernels at a smaller size.
4. **Match the target runtime.** Choose only among formats with an implemented loader, operator, and supported shapes on the deployment stack.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 2
LESSON_TITLE = 'The Sparsity Granularity Spectrum: Weights, Channels, Blocks, and N:M'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260810
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | original dense matrix and unstructured 50% magnitude mask |
| Candidate | block mask, exact 2:4 mask, and a physically narrowed dense matrix |
| Held constant | source weights, input batch, dtype, target zero budget, and timing method |
| Measurements | global sparsity, 2:4 compliance, physical shape, and median latency |
| Evidence | `pytorch-gpu` |

**Experiment:** Construct four 50%-budget representations and compare compliance, shape, and ordinary dense CUDA timing.


## 5. Read the experiment code

Each mask is generated explicitly so its local structure can be inspected. The experiment intentionally multiplies masked tensors through the ordinary dense PyTorch path; it does not claim cuSPARSELt dispatch. The narrow candidate changes the contracted work and provides a useful control for the claim that structure, not zero count, is what the kernel sees.

Do not execute until the code implements the frozen table above.


In [2]:
dtype = torch.bfloat16
rows, cols, batch = 2048, 2048, 32
w = torch.randn(rows, cols, device=DEVICE, dtype=dtype)
x = torch.randn(batch, cols, device=DEVICE, dtype=dtype)
dense = w.clone()
unstructured = w * magnitude_mask(w, 0.50)
blocks = w.reshape(rows // 16, 16, cols // 16, 16).clone()
block_selector = torch.arange((rows // 16) * (cols // 16), device=DEVICE).reshape(rows // 16, cols // 16) % 2
blocks = blocks * block_selector[:, None, :, None]
block_sparse = blocks.reshape_as(w)
nm = w * exact_2_4_mask(w)
narrow = w[: rows // 2]

td = timing_summary(cuda_times(lambda: F.linear(x, dense)))
tu = timing_summary(cuda_times(lambda: F.linear(x, unstructured)))
tb = timing_summary(cuda_times(lambda: F.linear(x, block_sparse)))
tnm = timing_summary(cuda_times(lambda: F.linear(x, nm)))
tn = timing_summary(cuda_times(lambda: F.linear(x, narrow)))
metrics = {
    "unstructured_sparsity": zero_fraction(unstructured),
    "block_sparsity": zero_fraction(block_sparse),
    "nm_sparsity": zero_fraction(nm),
    "unstructured_2_4_compliance": compliance_2_4(unstructured),
    "block_2_4_compliance": compliance_2_4(block_sparse),
    "nm_compliance": compliance_2_4(nm),
    "dense_median_ms": td["median_ms"],
    "unstructured_median_ms": tu["median_ms"],
    "block_median_ms": tb["median_ms"],
    "nm_dense_path_median_ms": tnm["median_ms"],
    "narrow_median_ms": tn["median_ms"],
    "narrow_shape": list(narrow.shape),
}
analysis = (
    f"All three masks were near 50% sparse, but exact 2:4 compliance was "
    f"{metrics['nm_compliance']:.1%} versus {metrics['unstructured_2_4_compliance']:.1%} for the "
    f"unstructured mask. The ordinary dense path measured {metrics['dense_median_ms']:.6f} ms for dense, "
    f"{metrics['unstructured_median_ms']:.6f} ms for unstructured, and "
    f"{metrics['nm_dense_path_median_ms']:.6f} ms for compliant values. Only the narrow control changed "
    "the matrix shape; no sparse-kernel dispatch is inferred from these timings."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Unstructured sparsity | 50.00% |
| 2:4 compliance | 100.00% |
| Dense median | 0.017920 ms |
| Unstructured median | 0.017888 ms |
| 2:4 dense-path median | 0.018384 ms |
| Narrow median | 0.018480 ms |


## 7. Interpret rather than merely print

All three masks were near 50% sparse, but exact 2:4 compliance was 100.0% versus 37.5% for the unstructured mask. The ordinary dense path measured 0.017920 ms for dense, 0.017888 ms for unstructured, and 0.018384 ms for compliant values. Only the narrow control changed the matrix shape; no sparse-kernel dispatch is inferred from these timings.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 2,
    "title": 'The Sparsity Granularity Spectrum: Weights, Channels, Blocks, and N:M',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Sparsity granularity is an interface between optimization and execution; a global zero rate is only one field of that interface.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 2,
  "title": "The Sparsity Granularity Spectrum: Weights, Channels, Blocks, and N:M",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260810
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "unstructured_sparsity": 0.5,
    "block_sparsity": 0.5,
    "nm_sparsity": 0.5,
    "unstructured_2_4_compliance": 0.37503910064697266,
    "block_2_4_compliance": 0.0,
    "nm_compliance": 1.0,
    "dense_median_ms": 0.017920000478625298,
    "unstructured_median_ms": 0.01788800023496151,
    "block_median_ms": 0.01809599995613098,
    "nm_dense_path_median_ms": 0.018384000286459923,
    "narrow_median_ms": 0.01848000008612871,
    "narrow_shape": [
      1024,
      2048
    ]
  },
  "analysis": "All three masks were near 50% sparse, but exact 2:4 compliance was 100.0% versus 37.5% for the unstructured mask. The ordinary dense path measured 

## 9. Make the bounded decision

> Sparsity granularity is an interface between optimization and execution; a global zero rate is only one field of that interface.

**Acceptance/rollback:** Select a granularity only after the target runtime's supported patterns and the model's accuracy sensitivity are both written down.

**Failure analysis:** A 2:4-compliant tensor can still use a dense tactic when it is not compressed into the required backend format, dtype, alignment, or build flag. A channel-pruned tensor can also be slower at awkward widths. Compliance is necessary for some paths, never sufficient for speed.


## 10. Extend the evidence

Run the compliant matrix through cuSPARSELt or TensorRT, capture its tactic log, and sweep dimensions around alignment boundaries while holding the nonzero budget fixed.

The full evidence boundary and references are in [`README.md`](README.md).
